# Note: this file has sence been split into two (data prep and plotting) as 31 and 32


# Figure 3 Precipitation Percentile Statistical Analysis

## Input Data
- **Zarr stores** with 15-minute precipitation period counts by percentile band
- Epochs: HIST (1990-2005) and EOC8.5/EOC4.5 (2085-2100)
- Temporal: Annual + 4 seasons (MAM, JJA, SON, DJF)
- Spatial: Full WRF-BCC domain (3.75 km, 899×1399 grid cells)
- Filtered to: **99th-100th percentile band**
- Format: Compressed Zarr format (~70% smaller than NetCDF)

## Processing Steps

1. **Calculate departures and percentages**:
   - `departure = future_mean - historical_mean`
   - `percentage = (departure / historical_mean) × 100`

2. **Statistical significance testing**:
   - **Mann-Whitney U test** (two-sided) gridcell by gridcell
   - **FDR correction** (Benjamini-Hochberg) to control for multiple comparisons
   - Significance threshold: α = 0.05

3. **Visualization**:
   - Coarsen 21×21 for figure readability
   - Stippling added where FDR-corrected p < α

## Output

- **Multi-panel figure** (5 rows × 3 columns): Annual + 4 seasons
  - Column 1: Historical climatology (count)
  - Column 2: Absolute departure (with significance stippling)
  - Column 3: Percentage change
- **Cached statistical files** (NetCDF): MWU raw p-values and FDR-corrected results

In [1]:
# imports and Setup
import os
import numpy as np
import xarray as xr
import pandas as pd
from scipy.stats import mannwhitneyu
from scipy.ndimage.filters import gaussian_filter
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.offsetbox import AnchoredText
from matplotlib.colors import ListedColormap
from matplotlib import gridspec, colors, colormaps, cm, ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cartopy.io.shapereader as shpreader
from distributed import Client
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm
import warnings

# Suppress common warnings for cleaner output
#warnings.filterwarnings('ignore', category=RuntimeWarning)

print("Initializing Dask client...")
# Set up Dask client
client = Client(n_workers=4, threads_per_worker=4, memory_limit="32GB")
print(f"Dask client initialized: {client}")


xr.set_options(display_style="html")

/tmp/ipykernel_19009/1535876927.py:7: DeprecationWarning: Please use `gaussian_filter` from the `scipy.ndimage` namespace, the `scipy.ndimage.filters` namespace is deprecated.
  from scipy.ndimage.filters import gaussian_filter


Initializing Dask client...


/anaconda3/envs/pyEAE/lib/python3.9/site-packages/distributed/node.py:179: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46307 instead
  warnings.warn(
2026-01-09 20:03:52,761 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-0iphplml', purging
2026-01-09 20:03:52,762 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-iadoeimc', purging
2026-01-09 20:03:52,763 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-9pm32ead', purging
2026-01-09 20:03:52,763 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-9af0ygl7', purging


Dask client initialized: <Client: 'tcp://127.0.0.1:34208' processes=4 threads=16, memory=119.21 GiB>


In [2]:
# Define paths and parameters
RCP_scenario = '8p5'  # options: "4p5" or "8p5"
if RCP_scenario not in ('4p5', '8p5'):
    raise ValueError(f"Invalid RCP_scenario value: {RCP_scenario} \t options: 4p5 or 8p5")

print("\nDefining file paths and parameters...")

# Updated paths to use Zarr stores instead of NetCDF
base_path_annual = '/home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/combined_datasets'
base_path_seasonal = '/home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/combined_seasonal_datasets'
path_grid = r"/home/sleake/WRF-BCC_Geogrid"

percentile_band = '99th-100th'

# Statistical analysis output directory - keep same location for cached stats
stats_output_dir = f'/home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC{RCP_scenario}/epoch_aggregated/statistical_analysis'

# Set desired significance level for stippling on the final figure
alpha = 0.01  # or 0.01
sig_display_threshold = 0.10  # e.g., 0.10 (10%), 0.25 (25%), 0.50 (50%)

# Control flags
recalculate_stats = False
test_mode = False
test_size = 100

print(f"Annual data path: {base_path_annual}")
print(f"Seasonal data path: {base_path_seasonal}")
print(f"Grid data path: {path_grid}")
print(f"Statistical output directory: {stats_output_dir}")
print(f"Using percentile band: {percentile_band}")
print(f"Data format: Zarr stores (compressed)")
print(f"Recalculate statistics flag: {recalculate_stats}")
print(f"Test mode: {test_mode}")

if test_mode:
    print(f"  Test domain size: {test_size}x{test_size} grid cells from center")
    stats_output_dir = os.path.join(stats_output_dir, 'test_runs')
    print(f"  Test output directory: {stats_output_dir}")

# Define file paths for historical and future data using Zarr
# Annual data from combined_datasets
hist_paths = {
    'annual': f'{base_path_annual}/dist_hist_dataset.zarr',
    'mam': f'{base_path_seasonal}/dist_hist_MAM_dataset.zarr',
    'jja': f'{base_path_seasonal}/dist_hist_JJA_dataset.zarr',
    'son': f'{base_path_seasonal}/dist_hist_SON_dataset.zarr',
    'djf': f'{base_path_seasonal}/dist_hist_DJF_dataset.zarr'
}

futr_paths = {
    'annual': f'{base_path_annual}/dist_eoc{RCP_scenario}_dataset.zarr',
    'mam': f'{base_path_seasonal}/dist_eoc{RCP_scenario}_MAM_dataset.zarr',
    'jja': f'{base_path_seasonal}/dist_eoc{RCP_scenario}_JJA_dataset.zarr',
    'son': f'{base_path_seasonal}/dist_eoc{RCP_scenario}_SON_dataset.zarr',
    'djf': f'{base_path_seasonal}/dist_eoc{RCP_scenario}_DJF_dataset.zarr'
}

print(f"Defined paths for {len(hist_paths)} datasets each for historical and future scenarios.")
print("Note: Using Zarr format vs earlier NetCDF- ~70% smaller than NetCDF with same data")


Defining file paths and parameters...
Annual data path: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/combined_datasets
Seasonal data path: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/combined_seasonal_datasets
Grid data path: /home/sleake/WRF-BCC_Geogrid
Statistical output directory: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC8p5/epoch_aggregated/statistical_analysis
Using percentile band: 99th-100th
Data format: Zarr stores (compressed)
Recalculate statistics flag: False
Test mode: False
Defined paths for 5 datasets each for historical and future scenarios.
Note: Using Zarr format vs earlier NetCDF- ~70% smaller than NetCDF with same data


In [3]:
# Load data and add coordinates
def load_datasets():
    """Load historical and future datasets with progress tracking"""
    print("\nLoading datasets...")
    hist_datasets = {}
    futr_datasets = {}
    
    # Load historical datasets
    print("Loading historical datasets:")
    for season, path in tqdm(hist_paths.items(), desc="Historical data"):
        print(f"  Loading {season.upper()} historical data from: {os.path.basename(path)}")
        
        # Use xr.open_dataset for Zarr with older xarray, or xr.open_zarr for newer versions
        try:
            ds = xr.open_dataset(path, chunks='auto', engine='zarr')
        except ValueError:
            # Fallback for older xarray that doesn't support engine='zarr'
            import zarr
            store = zarr.DirectoryStore(path)
            ds = xr.open_zarr(store)
        
        print(f"    Dataset shape: {dict(ds.sizes)}")
        
        # Filter by percentile band - FIX: Use .sel() instead of .where() for dask arrays
        if 'percentile_band' in ds.coords:
            try:
                ds_filtered = ds.sel(percentile_band=percentile_band)
                print(f"    Filtered to percentile band {percentile_band}")
            except (KeyError, ValueError):
                print(f"    WARNING: Percentile band {percentile_band} not found, using all data")
                ds_filtered = ds
        elif 'percentile_label' in ds.coords:
            try:
                ds_filtered = ds.sel(percentile_label=percentile_band)
                print(f"    Filtered to percentile label {percentile_band}")
            except (KeyError, ValueError):
                print(f"    Using computed boolean filter for percentile_label...")
                mask = (ds['percentile_label'] == percentile_band).compute()
                valid_indices = mask.values.nonzero()[0]
                if len(valid_indices) > 0:
                    ds_filtered = ds.isel(concat_dim=valid_indices)
                    print(f"    Filtered to {len(valid_indices)} matching records")
                else:
                    print(f"    WARNING: No matching records found for {percentile_band}")
                    ds_filtered = ds
        else:
            ds_filtered = ds
            print(f"    No percentile band filtering needed")
        
        # NORMALIZE DIMENSION NAMES (concat_dim -> Time)
        ds_filtered = normalize_dimension_names(ds_filtered)
        
        # Apply test mode subsetting if enabled
        if test_mode:
            ds_filtered = subset_for_testing(ds_filtered, season, 'historical')
        
        hist_datasets[season] = ds_filtered
    
    # Load future datasets
    print("\nLoading future datasets:")
    for season, path in tqdm(futr_paths.items(), desc="Future data"):
        print(f"  Loading {season.upper()} future data from: {os.path.basename(path)}")
        
        try:
            ds = xr.open_dataset(path, chunks='auto', engine='zarr')
        except ValueError:
            import zarr
            store = zarr.DirectoryStore(path)
            ds = xr.open_zarr(store)
        
        print(f"    Dataset shape: {dict(ds.sizes)}")
        
        # Filter by percentile band
        if 'percentile_band' in ds.coords:
            try:
                ds_filtered = ds.sel(percentile_band=percentile_band)
                print(f"    Filtered to percentile band {percentile_band}")
            except (KeyError, ValueError):
                print(f"    WARNING: Percentile band {percentile_band} not found, using all data")
                ds_filtered = ds
        elif 'percentile_label' in ds.coords:
            try:
                ds_filtered = ds.sel(percentile_label=percentile_band)
                print(f"    Filtered to percentile label {percentile_band}")
            except (KeyError, ValueError):
                print(f"    Using computed boolean filter for percentile_label...")
                mask = (ds['percentile_label'] == percentile_band).compute()
                valid_indices = mask.values.nonzero()[0]
                if len(valid_indices) > 0:
                    ds_filtered = ds.isel(concat_dim=valid_indices)
                    print(f"    Filtered to {len(valid_indices)} matching records")
                else:
                    print(f"    WARNING: No matching records found for {percentile_band}")
                    ds_filtered = ds
        else:
            ds_filtered = ds
            print(f"    No percentile band filtering needed")
        
        # NORMALIZE DIMENSION NAMES (concat_dim -> Time)
        ds_filtered = normalize_dimension_names(ds_filtered)
        
        # Apply test mode subsetting if enabled
        if test_mode:
            ds_filtered = subset_for_testing(ds_filtered, season, 'future')
        
        futr_datasets[season] = ds_filtered
    
    # Add lat/lon coordinates from grid file
    print("\nLoading grid coordinates...")
    grid_file = os.path.join(path_grid, "geo_em.d01.nc")
    print(f"Loading grid from: {grid_file}")
    
    if not os.path.exists(grid_file):
        raise FileNotFoundError(f"Grid file not found: {grid_file}")
    
    grid = xr.open_dataset(grid_file)
    lat_coords = grid.XLAT_M[0, :, :].values
    lon_coords = grid.XLONG_M[0, :, :].values
    
    # Apply test mode subsetting to coordinates if enabled
    if test_mode:
        lat_coords, lon_coords = subset_coordinates_for_testing(lat_coords, lon_coords)
    
    print(f"Grid coordinates shape: lat {lat_coords.shape}, lon {lon_coords.shape}")
    
    # Add coordinates to all datasets
    print("Adding lat/lon coordinates to datasets...")
    all_datasets = [hist_datasets, futr_datasets]
    for dataset_dict, name in zip(all_datasets, ['Historical', 'Future']):
        print(f"  Processing {name} datasets...")
        for season, ds in dataset_dict.items():
            ds.coords['lat'] = (['south_north', 'west_east'], lat_coords)
            ds.coords['lon'] = (['south_north', 'west_east'], lon_coords)
            print(f"    Added coordinates to {season.upper()}")
    
    print("Dataset loading complete.")
    print(f"Zarr format benefits: ~70% size reduction, faster access, cloud-ready")
    
    return hist_datasets, futr_datasets

def normalize_dimension_names(ds):
    """
    Normalize dimension names to match expected naming convention.
    Zarr files use 'concat_dim' while NetCDF uses 'Time'.
    """
    # Rename concat_dim to Time if it exists
    if 'concat_dim' in ds.dims:
        ds = ds.rename({'concat_dim': 'Time'})
        print(f"      Renamed 'concat_dim' to 'Time'")
    
    return ds

def subset_for_testing(ds, season, data_type):
    """Subset dataset to test domain (center of domain)"""
    print(f"    Applying test mode subset for {season.upper()} {data_type}...")
    
    # Get original dimensions
    original_sn = ds.sizes['south_north']
    original_we = ds.sizes['west_east']
    
    # Calculate center indices
    center_sn = original_sn // 2
    center_we = original_we // 2
    
    # Calculate subset bounds (test_size x test_size around center)
    half_test = test_size // 2
    sn_start = max(0, center_sn - half_test)
    sn_end = min(original_sn, center_sn + half_test)
    we_start = max(0, center_we - half_test)
    we_end = min(original_we, center_we + half_test)
    
    # Apply subset
    ds_subset = ds.isel(south_north=slice(sn_start, sn_end), 
                       west_east=slice(we_start, we_end))
    
    actual_sn = sn_end - sn_start
    actual_we = we_end - we_start
    
    print(f"      Original domain: {original_sn}x{original_we}")
    print(f"      Test subset: {actual_sn}x{actual_we} (indices: [{sn_start}:{sn_end}, {we_start}:{we_end}])")
    print(f"      Reduction factor: {original_sn*original_we // (actual_sn*actual_we):.1f}x smaller")
    
    return ds_subset

def subset_coordinates_for_testing(lat_coords, lon_coords):
    """Subset coordinate arrays to match test domain"""
    print("    Applying test mode subset to coordinates...")
    
    # Get original dimensions
    original_sn, original_we = lat_coords.shape
    
    # Calculate center indices
    center_sn = original_sn // 2
    center_we = original_we // 2
    
    # Calculate subset bounds
    half_test = test_size // 2
    sn_start = max(0, center_sn - half_test)
    sn_end = min(original_sn, center_sn + half_test)
    we_start = max(0, center_we - half_test)
    we_end = min(original_we, center_we + half_test)
    
    # Apply subset to coordinates
    lat_subset = lat_coords[sn_start:sn_end, we_start:we_end]
    lon_subset = lon_coords[sn_start:sn_end, we_start:we_end]
    
    print(f"      Coordinate subset: {lat_subset.shape}")
    
    return lat_subset, lon_subset

print("Executing dataset loading...")
hist_ds, futr_ds = load_datasets()
print(f"Successfully loaded {len(hist_ds)} historical and {len(futr_ds)} future datasets.")

print("Historical annual dataset info:")

# Check what dimensions remain after filtering
print(f"Dimensions: {dict(hist_ds['annual'].dims)}")
print(f"Coordinates: {list(hist_ds['annual'].coords)}")
print(f"Data variables: {list(hist_ds['annual'].data_vars.keys())}")

# Check the data variable (PRECIP_PERIOD)
if 'PRECIP_PERIOD' in hist_ds['annual'].data_vars:
    data_var = hist_ds['annual'].PRECIP_PERIOD
    print(f"\nPRECIP_PERIOD info:")
    print(f"  Shape: {data_var.shape}")
    print(f"  Dimensions: {data_var.dims}")
    print(f"  Data type: {data_var.dtype}")
    
    # Check for non-zero data
    print(f"\nData statistics:")
    print(f"  Min: {data_var.min().compute().values}")
    print(f"  Max: {data_var.max().compute().values}")
    print(f"  Mean: {data_var.mean().compute().values}")
    print(f"  Non-zero count: {(data_var > 0).sum().compute().values}")
    print(f"  Total cells: {data_var.size}")
else:
    print("WARNING: PRECIP_PERIOD variable not found")
    print(f"Available variables: {list(hist_ds['annual'].data_vars.keys())}")

print(f"\nNote: Dataset was filtered to percentile band '{percentile_band}'")
print(f"The percentile_band dimension has been removed after filtering to a single band")


Executing dataset loading...

Loading datasets...
Loading historical datasets:


Historical data:   0%|          | 0/5 [00:00<?, ?it/s]

  Loading ANNUAL historical data from: dist_hist_dataset.zarr
    Dataset shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Using computed boolean filter for percentile_label...
      Renamed 'concat_dim' to 'Time'
  Loading MAM historical data from: dist_hist_MAM_dataset.zarr
    Dataset shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Using computed boolean filter for percentile_label...
      Renamed 'concat_dim' to 'Time'
  Loading JJA historical data from: dist_hist_JJA_dataset.zarr
    Dataset shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Using computed boolean filter for percentile_label...
      Renamed 'concat_dim' to 'Time'
  Loading SON historical data from: dist_hist_SON_dataset.zarr
    Dataset shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Using computed boolean filter for percentile_label...
      Renamed 'concat_dim' to 'Time'
  Loading DJF historical data from: dist_hist_DJF_dataset

Future data:   0%|          | 0/5 [00:00<?, ?it/s]

  Loading ANNUAL future data from: dist_eoc8p5_dataset.zarr
    Dataset shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Using computed boolean filter for percentile_label...
      Renamed 'concat_dim' to 'Time'
  Loading MAM future data from: dist_eoc8p5_MAM_dataset.zarr
    Dataset shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Using computed boolean filter for percentile_label...
      Renamed 'concat_dim' to 'Time'
  Loading JJA future data from: dist_eoc8p5_JJA_dataset.zarr
    Dataset shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Using computed boolean filter for percentile_label...
      Renamed 'concat_dim' to 'Time'
  Loading SON future data from: dist_eoc8p5_SON_dataset.zarr
    Dataset shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Using computed boolean filter for percentile_label...
      Renamed 'concat_dim' to 'Time'
  Loading DJF future data from: dist_eoc8p5_DJF_dataset.zarr
    

In [4]:
# Load mask (if we are spatially masking out our data for a more regionally focused plot)
print("\nLoading spatial mask...")
mask_path = "/home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/WRF-BCC_Geogrid"
mask_file = os.path.join(mask_path, "ECON_mask.nc")
print(f"Loading mask from: {mask_file}")

if not os.path.exists(mask_file):
    raise FileNotFoundError(f"Mask file not found: {mask_file}")

mask = xr.open_dataset(mask_file)
print(f"Mask loaded with shape: {dict(mask.sizes)}")
print(f"Mask variables: {list(mask.data_vars.keys())}")



Loading spatial mask...
Loading mask from: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/WRF-BCC_Geogrid/ECON_mask.nc
Mask loaded with shape: {'south_north': 899, 'west_east': 1399}
Mask variables: ['__xarray_dataarray_variable__']


In [5]:
# Utility functions
print("\nDefining utility functions...")

def downsample_ds(ds, aggregation_factor=(1,1), reducer='mean'):
    """Resample data to a larger grid for faster computation with progress tracking"""
    print(f"    Downsampling with factor {aggregation_factor} using {reducer} method...")
    boundary = "trim"
    
    if reducer not in ['mean', 'min', 'max', 'var', 'std']:
        raise ValueError(f"Reducer must be one of ['mean', 'min', 'max', 'var', 'std']")
    
    original_shape = dict(ds.sizes)
    
    # Coarsen both dimensions at once
    coarsened = ds.coarsen(
        west_east=aggregation_factor[0], 
        south_north=aggregation_factor[1], 
        boundary=boundary
    )
    
    # Apply the reduction method directly
    if reducer == 'mean':
        ds_downsampled = coarsened.mean(skipna=True)
    elif reducer == 'min':
        ds_downsampled = coarsened.min(skipna=True)
    elif reducer == 'max':
        ds_downsampled = coarsened.max(skipna=True)
    elif reducer == 'var':
        ds_downsampled = coarsened.var(skipna=True)
    elif reducer == 'std':
        ds_downsampled = coarsened.std(skipna=True)
    
    new_shape = dict(ds_downsampled.sizes)
    print(f"    Downsampled from {original_shape} to {new_shape}")
    
    return ds_downsampled

def apply_mannwhitneyu(data0, data1):
    """Apply Mann-Whitney U test element-wise with progress tracking"""
    print(f"    Applying Mann-Whitney U test...")
    print(f"    Data0 shape: {data0.shape}, Data1 shape: {data1.shape}")
    
    # Check if we have any data to work with
    if data0.size == 0 or data1.size == 0:
        print(f"    WARNING: Empty dataset detected!")
        print(f"    Data0 size: {data0.size}, Data1 size: {data1.size}")
        print(f"    This may be due to percentile band filtering removing all data")
        print(f"    Returning NaN-filled result with spatial dimensions...")
        
        # Create NaN-filled result with proper spatial dimensions
        # Get spatial shape from the last two dimensions
        if len(data0.shape) >= 2:
            spatial_shape = data0.shape[-2:]
        elif len(data1.shape) >= 2:
            spatial_shape = data1.shape[-2:]
        else:
            print(f"    ERROR: Cannot determine spatial dimensions from data shapes")
            spatial_shape = (100, 100)  # Fallback shape
        
        p_value_2d = np.full(spatial_shape, np.nan)
        print(f"    Created NaN-filled result with shape: {p_value_2d.shape}")
        
        return xr.Dataset({'p_value': (('south_north', 'west_east'), p_value_2d)})
    
    # Check for dimension mismatch in the middle dimensions
    if data0.shape[1] == 0 or data1.shape[1] == 0:
        print(f"    WARNING: Zero-sized middle dimension detected!")
        print(f"    This suggests no data remains after percentile band filtering")
        print(f"    Data0 middle dim: {data0.shape[1]}, Data1 middle dim: {data1.shape[1]}")
        
        # Get spatial dimensions from the last two dimensions  
        spatial_shape = (data0.shape[-2], data0.shape[-1])
        p_value_2d = np.full(spatial_shape, np.nan)
        print(f"    Created NaN-filled result with shape: {p_value_2d.shape}")
        
        return xr.Dataset({'p_value': (('south_north', 'west_east'), p_value_2d)})
    
    flat_data0 = data0.values.reshape((data0.shape[0], -1))
    flat_data1 = data1.values.reshape((data1.shape[0], -1))
    
    print(f"    Flattened data shapes: {flat_data0.shape}, {flat_data1.shape}")
    
    # Additional check after flattening
    if flat_data0.shape[1] == 0 or flat_data1.shape[1] == 0:
        print(f"    WARNING: No grid points to process after flattening!")
        spatial_shape = data0.shape[-2:] if len(data0.shape) >= 2 else (100, 100)
        p_value_2d = np.full(spatial_shape, np.nan)
        print(f"    Created NaN-filled result with shape: {p_value_2d.shape}")
        
        return xr.Dataset({'p_value': (('south_north', 'west_east'), p_value_2d)})
    
    stat = np.empty_like(flat_data0[0], dtype=float)
    p_value = np.empty_like(flat_data0[0], dtype=float)

    print(f"    Computing Mann-Whitney U for {len(stat)} grid points...")
    for i in tqdm(range(len(stat)), desc="    MWU test progress", leave=False):
        try:
            _, p_value[i] = mannwhitneyu(flat_data0[:, i], flat_data1[:, i], alternative='two-sided')
        except Exception as e:
            print(f"    Warning: MWU test failed for point {i}: {e}")
            p_value[i] = np.nan

    # Reshape to original spatial dimensions (remove any singleton dimensions)
    original_spatial_shape = data0.shape[1:]  # Skip time dimension
    p_value_reshaped = p_value.reshape(original_spatial_shape)
    
    # Squeeze out any singleton dimensions to get 2D spatial grid
    p_value_2d = np.squeeze(p_value_reshaped)
    
    print(f"    Mann-Whitney U test completed. Result shape: {p_value_2d.shape}")
    print(f"    Expected dimensions: south_north x west_east")
    
    # Handle edge cases for dimensions
    if p_value_2d.ndim == 0:
        print(f"    WARNING: Got 0D result, converting to 2D NaN array")
        spatial_shape = data0.shape[-2:] if len(data0.shape) >= 2 else (100, 100)
        p_value_2d = np.full(spatial_shape, np.nan)
    elif p_value_2d.ndim == 1:
        print(f"    WARNING: Got 1D result, attempting to reshape to 2D")
        # Try to reshape to the expected spatial dimensions
        spatial_shape = data0.shape[-2:]
        if np.prod(spatial_shape) == p_value_2d.size:
            p_value_2d = p_value_2d.reshape(spatial_shape)
        else:
            print(f"    ERROR: Cannot reshape 1D result to expected spatial shape")
            p_value_2d = np.full(spatial_shape, np.nan)
    elif p_value_2d.ndim > 2:
        print(f"    WARNING: Got {p_value_2d.ndim}D result, attempting to squeeze to 2D")
        # Try additional squeezing
        while p_value_2d.ndim > 2 and 1 in p_value_2d.shape:
            p_value_2d = np.squeeze(p_value_2d)
        
        if p_value_2d.ndim != 2:
            print(f"    ERROR: Still have {p_value_2d.ndim}D result after squeezing")
            print(f"    Shape: {p_value_2d.shape}")
            # Create a proper 2D array
            spatial_shape = data0.shape[-2:] if len(data0.shape) >= 2 else (100, 100)
            p_value_2d = np.full(spatial_shape, np.nan)
            print(f"    Created fallback NaN-filled result: {p_value_2d.shape}")
    
    # Final validation
    if p_value_2d.ndim != 2:
        print(f"    ERROR: Final result is not 2D: {p_value_2d.ndim}D with shape {p_value_2d.shape}")
        print(f"    Creating fallback 2D NaN array")
        spatial_shape = data0.shape[-2:] if len(data0.shape) >= 2 else (100, 100)
        p_value_2d = np.full(spatial_shape, np.nan)
    
    print(f"    Final result shape: {p_value_2d.shape}")
    
    return xr.Dataset({'p_value': (('south_north', 'west_east'), p_value_2d)})

def fdr_adjust(ds):
    """Apply False Discovery Rate correction"""
    print(f"    Applying FDR correction...")
    p_values = ds.p_value.values.flatten()
    
    # Remove NaN values for correction
    valid_mask = ~np.isnan(p_values)
    valid_p_values = p_values[valid_mask]
    
    print(f"    Processing {len(valid_p_values)} valid p-values out of {len(p_values)} total")
    
    if len(valid_p_values) == 0:
        print("    Warning: No valid p-values found for FDR correction")
        corrected_p_values = p_values
    else:
        reject, corrected_valid, _, _ = multipletests(valid_p_values, method='fdr_bh')
        
        # Put corrected values back into full array
        corrected_p_values = p_values.copy()
        corrected_p_values[valid_mask] = corrected_valid
    
    corrected_p_values = corrected_p_values.reshape(ds.p_value.shape)
    
    ds_fdr_corrected = xr.Dataset({
        'p_value_fdr_corrected': (('south_north', 'west_east'), corrected_p_values)
    })
    
    ds_fdr_corrected.coords['lat'] = ds.lat
    ds_fdr_corrected.coords['lon'] = ds.lon
    
    print(f"    FDR correction completed.")
    return ds_fdr_corrected

print("Utility functions defined.")


Defining utility functions...
Utility functions defined.


In [6]:
# Statistical analysis with caching
def check_and_create_stats_directory():
    """Create statistical analysis directory if it doesn't exist"""
    if not os.path.exists(stats_output_dir):
        print(f"Creating statistical analysis directory: {stats_output_dir}")
        os.makedirs(stats_output_dir, exist_ok=True)
    else:
        print(f"Statistical analysis directory exists: {stats_output_dir}")

def get_stats_file_paths():
    """Generate file paths for saved statistical results"""
    stats_files = {}
    seasons = ['annual', 'mam', 'jja', 'son', 'djf']
    
    for season in seasons:
        # Add test mode suffix to filenames
        test_suffix = f"_TEST{test_size}x{test_size}" if test_mode else ""
        
        # MWU test results
        mwu_file = os.path.join(stats_output_dir, f'MWU_HIST-EOC8p5_{percentile_band}_WetPeriod_0p254base_{season}{test_suffix}.nc')
        # FDR corrected results
        fdr_file = os.path.join(stats_output_dir, f'MWU_FDR_HIST-EOC8p5_{percentile_band}_WetPeriod_0p254base_{season}{test_suffix}.nc')
        
        stats_files[season] = {
            'mwu': mwu_file,
            'fdr': fdr_file
        }
    
    return stats_files

def check_stats_files_exist(stats_files):
    """Check which statistical files already exist"""
    existing_files = {}
    missing_files = {}
    
    for season, files in stats_files.items():
        existing_files[season] = {}
        missing_files[season] = {}
        
        for test_type, filepath in files.items():
            if os.path.exists(filepath):
                existing_files[season][test_type] = filepath
                print(f"  Found existing {test_type.upper()} file for {season.upper()}: {os.path.basename(filepath)}")
            else:
                missing_files[season][test_type] = filepath
                print(f"  Missing {test_type.upper()} file for {season.upper()}: {os.path.basename(filepath)}")
    
    return existing_files, missing_files

def load_existing_stats(existing_files):
    """Load existing statistical results from files"""
    print("Loading existing statistical results...")
    mwu_tests = {}
    fdr_tests = {}
    
    for season, files in existing_files.items():
        if 'mwu' in files:
            print(f"  Loading MWU results for {season.upper()}...")
            mwu_tests[season] = xr.open_dataset(files['mwu'])
            
        if 'fdr' in files:
            print(f"  Loading FDR results for {season.upper()}...")
            fdr_tests[season] = xr.open_dataset(files['fdr'])
    
    return mwu_tests, fdr_tests

def save_stats_results(mwu_tests, fdr_tests, stats_files):
    """Save statistical results to files"""
    print("Saving statistical results...")
    
    for season in mwu_tests.keys():
        # Save MWU results
        mwu_file = stats_files[season]['mwu']
        print(f"  Saving MWU results for {season.upper()} to: {os.path.basename(mwu_file)}")
        
        # Add metadata to MWU results (convert boolean to string for NetCDF compatibility)
        mwu_tests[season].attrs.update({
            'description': f'Mann-Whitney U test results for {season} precipitation periods',
            'comparison': f'HIST vs EOC{RCP_Scenario}',
            'percentile_band': percentile_band,
            'test_type': 'Mann-Whitney U (two-sided)',
            'created_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
            'null_hypothesis': f'No difference in distributions between HIST and EOC{RCP_Scenario}',
            'test_mode': str(test_mode),  # Convert boolean to string
            'test_domain_size': f"{test_size}x{test_size}" if test_mode else "full_domain"
        })
        
        mwu_tests[season].to_netcdf(mwu_file)
        
        # Save FDR results if available
        if season in fdr_tests:
            fdr_file = stats_files[season]['fdr']
            print(f"  Saving FDR results for {season.upper()} to: {os.path.basename(fdr_file)}")
            
            # Add metadata to FDR results (convert boolean to string for NetCDF compatibility)
            fdr_tests[season].attrs.update({
                'description': f'FDR-corrected Mann-Whitney U test results for {season} precipitation periods',
                'comparison': 'HIST vs EOC4.5',
                'percentile_band': percentile_band,
                'correction_method': 'Benjamini-Hochberg FDR',
                'created_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
                'significance_threshold': 0.01,
                'test_mode': str(test_mode),  # Convert boolean to string
                'test_domain_size': f"{test_size}x{test_size}" if test_mode else "full_domain"
            })
            
            fdr_tests[season].to_netcdf(fdr_file)

def calculate_departures_and_stats():
    """Calculate departures, percentages, and statistical tests with caching capability"""
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS WITH CACHING")
    print("="*60)
    
    # Create stats directory if needed
    check_and_create_stats_directory()
    
    # Get file paths for statistical results
    stats_files = get_stats_file_paths()
    
    # Check which files exist
    print("\nChecking for existing statistical files...")
    existing_files, missing_files = check_stats_files_exist(stats_files)
    
    # Determine if we need to calculate or can load existing results
    seasons = ['annual', 'mam', 'jja', 'son', 'djf']
    need_calculation = recalculate_stats
    
    if not recalculate_stats:
        # Check if any files are missing
        for season in seasons:
            if season not in existing_files or 'mwu' not in existing_files[season]:
                need_calculation = True
                print(f"  Need to calculate statistics for {season.upper()} (missing files)")
                break
    else:
        print("  Recalculation flag is True - will recalculate all statistics")
    
    # Initialize results dictionaries
    departures = {}
    percentages = {}
    mwu_tests = {}
    fdr_tests = {}
    
    if not need_calculation:
        print("\nAll statistical files exist - loading from cache...")
        mwu_tests, fdr_tests = load_existing_stats(existing_files)
        print("Successfully loaded existing statistical results")
    else:
        print("\nCalculating new statistical results...")
        
    # Always calculate departures and percentages (quick operations)
    print("\nCalculating departures and percentages...")
    for season in tqdm(seasons, desc="Processing departures/percentages"):
        print(f"\n  Processing {season.upper()} data:")
        
        # Calculate mean departures
        print("    Calculating temporal means...")
        hist_mean = hist_ds[season].mean(dim="Time")
        futr_mean = futr_ds[season].mean(dim="Time")
        
        print("    Computing departures (future - historical)...")
        departures[season] = futr_mean - hist_mean
        
        # Calculate percentage changes
        print("    Computing percentage changes...")
        percentages[season] = ((departures[season]) / hist_mean) * 100
        
        print(f"    {season.upper()} departures/percentages complete.")
    
    # Calculate or load statistical tests
    if need_calculation:
        print("\nPerforming Mann-Whitney U tests...")
        for season in tqdm(seasons, desc="Statistical tests"):
            print(f"\n  Processing statistical tests for {season.upper()}:")
            
            # Perform Mann-Whitney U test
            print("    Performing Mann-Whitney U statistical test...")
            mwu_tests[season] = apply_mannwhitneyu(
                hist_ds[season].PRECIP_PERIOD, 
                futr_ds[season].PRECIP_PERIOD
            )
            # Add coordinates
            mwu_tests[season].coords['lat'] = hist_ds[season].coords['lat']
            mwu_tests[season].coords['lon'] = hist_ds[season].coords['lon']
            
            # Calculate FDR correction
            print("    Applying FDR correction...")
            fdr_tests[season] = fdr_adjust(mwu_tests[season])
            
            print(f"    {season.upper()} statistical tests complete.")
        
        # Save all results
        print("\nSaving statistical results to files...")
        save_stats_results(mwu_tests, fdr_tests, stats_files)
    
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS COMPLETE")
    print("="*60)
    
    return departures, percentages, mwu_tests, fdr_tests
print("Stats functions defined.")

Stats functions defined.


In [7]:
# Data Quality Tests
def run_data_quality_tests(hist_ds, futr_ds, departures, percentages, mwu_tests, fdr_tests=None):
    """Run data quality tests"""
    print("\n" + "="*60)
    print("RUNNING DATA QUALITY TESTS")
    print("="*60)
    
    seasons = ['annual', 'mam', 'jja', 'son', 'djf']
    test_results = {}
    
    for season in seasons:
        print(f"\nTesting {season.upper()} data:")
        season_results = {}
        
        # Test 1: Check for all-NaN datasets
        print("  Test 1: Checking for all-NaN datasets...")
        hist_all_nan = hist_ds[season].PRECIP_PERIOD.isnull().all().compute()
        futr_all_nan = futr_ds[season].PRECIP_PERIOD.isnull().all().compute()
        dep_all_nan = departures[season].PRECIP_PERIOD.isnull().all().compute()
        pct_all_nan = percentages[season].PRECIP_PERIOD.isnull().all().compute()
        
        season_results['all_nan_test'] = {
            'historical': bool(hist_all_nan),
            'future': bool(futr_all_nan),
            'departures': bool(dep_all_nan),
            'percentages': bool(pct_all_nan)
        }
        
        if any([hist_all_nan, futr_all_nan, dep_all_nan, pct_all_nan]):
            print("    WARNING: Found all-NaN datasets!")
            for name, is_nan in season_results['all_nan_test'].items():
                if is_nan:
                    print(f"      {name}: ALL NaN")
        else:
            print("    PASS: No all-NaN datasets found")
        
        # Test 2: Check data ranges
        print("  Test 2: Checking data ranges...")
        # Use xarray statistical methods instead of describe()
        hist_data = hist_ds[season].PRECIP_PERIOD
        futr_data = futr_ds[season].PRECIP_PERIOD
        
        # Calculate statistics manually for xarray
        hist_min = float(hist_data.min().compute()) if hist_data.size > 0 else np.nan
        hist_max = float(hist_data.max().compute()) if hist_data.size > 0 else np.nan
        hist_mean = float(hist_data.mean().compute()) if hist_data.size > 0 else np.nan
        
        futr_min = float(futr_data.min().compute()) if futr_data.size > 0 else np.nan
        futr_max = float(futr_data.max().compute()) if futr_data.size > 0 else np.nan
        futr_mean = float(futr_data.mean().compute()) if futr_data.size > 0 else np.nan
        
        season_results['data_ranges'] = {
            'hist_min': hist_min,
            'hist_max': hist_max,
            'hist_mean': hist_mean,
            'futr_min': futr_min,
            'futr_max': futr_max,
            'futr_mean': futr_mean
        }
        
        print(f"    Historical: min={season_results['data_ranges']['hist_min']:.2f}, "
              f"max={season_results['data_ranges']['hist_max']:.2f}, "
              f"mean={season_results['data_ranges']['hist_mean']:.2f}")
        print(f"    Future: min={season_results['data_ranges']['futr_min']:.2f}, "
              f"max={season_results['data_ranges']['futr_max']:.2f}, "
              f"mean={season_results['data_ranges']['futr_mean']:.2f}")
        
        # Test 3: Check for negative values (shouldn't exist in precipitation counts)
        print("  Test 3: Checking for negative precipitation counts...")
        hist_negative = (hist_ds[season].PRECIP_PERIOD < 0).any().compute()
        futr_negative = (futr_ds[season].PRECIP_PERIOD < 0).any().compute()
        
        season_results['negative_values'] = {
            'historical': bool(hist_negative),
            'future': bool(futr_negative)
        }
        
        if hist_negative or futr_negative:
            print("    WARNING: Found negative precipitation values!")
        else:
            print("    PASS: No negative values found")
        
        # Test 4: Check coordinate consistency
        print("  Test 4: Checking coordinate consistency...")
        lat_consistent = np.allclose(hist_ds[season].lat, futr_ds[season].lat, equal_nan=True)
        lon_consistent = np.allclose(hist_ds[season].lon, futr_ds[season].lon, equal_nan=True)
        
        season_results['coordinates'] = {
            'lat_consistent': lat_consistent,
            'lon_consistent': lon_consistent
        }
        
        if not (lat_consistent and lon_consistent):
            print("    WARNING: Inconsistent coordinates between historical and future data!")
        else:
            print("    PASS: Coordinates are consistent")
        
        # Test 5: Check Mann-Whitney U test results
        print("  Test 5: Checking statistical test results...")
        valid_pvals = (~np.isnan(mwu_tests[season].p_value.values)).sum()
        total_points = mwu_tests[season].p_value.size
        pval_coverage = valid_pvals / total_points * 100
        
        season_results['mwu_test'] = {
            'valid_pvals': int(valid_pvals),
            'total_points': int(total_points),
            'coverage_percent': float(pval_coverage)
        }
        
        print(f"    Statistical test coverage: {valid_pvals}/{total_points} ({pval_coverage:.1f}%)")
        
        if pval_coverage < 50:
            print("    INFO: Low statistical test coverage (Missing gridcell tests, check data dims)")
        else:
            print("    Info: Good statistical test coverage (all grid cells have MWU pvals)")
        
        # Test 6: Check FDR correction if available
        if fdr_tests and season in fdr_tests:
            print("  Test 6: Checking FDR correction results...")
            fdr_valid_pvals = (~np.isnan(fdr_tests[season].p_value_fdr_corrected.values)).sum()
            fdr_total_points = fdr_tests[season].p_value_fdr_corrected.size
            fdr_coverage = fdr_valid_pvals / fdr_total_points * 100
            
            # Count significant points before and after FDR correction
            raw_sig = (mwu_tests[season].p_value < 0.01).sum().compute()
            fdr_sig = (fdr_tests[season].p_value_fdr_corrected < 0.01).sum().compute()
            
            season_results['fdr_test'] = {
                'fdr_coverage_percent': float(fdr_coverage),
                'raw_significant_points': int(raw_sig),
                'fdr_significant_points': int(fdr_sig),
                'correction_factor': float(fdr_sig / raw_sig) if raw_sig > 0 else 0.0
            }
            
            print(f"    FDR correction coverage: {fdr_valid_pvals}/{fdr_total_points} ({fdr_coverage:.1f}%)")
            print(f"    Significant points: Raw={raw_sig}, FDR-corrected={fdr_sig} "
                  f"(factor={season_results['fdr_test']['correction_factor']:.3f})")
        else:
            print("  Test 6: FDR correction results not available")
        
        # Test 7: Check for extreme percentage changes
        print("  Test 7: Checking for extreme percentage changes...")
        pct_data = percentages[season].PRECIP_PERIOD
        
        # Calculate statistics manually for xarray
        pct_min = float(pct_data.min().compute()) if pct_data.size > 0 else np.nan
        pct_max = float(pct_data.max().compute()) if pct_data.size > 0 else np.nan
        
        extreme_positive = pct_max > 500 if not np.isnan(pct_max) else False  # More than 500% increase
        extreme_negative = pct_min < -95 if not np.isnan(pct_min) else False  # More than 95% decrease
        
        season_results['extreme_changes'] = {
            'max_pct_change': pct_max,
            'min_pct_change': pct_min,
            'extreme_positive': extreme_positive,
            'extreme_negative': extreme_negative
        }
        
        print(f"    Percentage change range: {pct_min:.1f}% to {pct_max:.1f}%")
        
        if extreme_positive or extreme_negative:
            print("    INFO: Found large percentage changes (> 95% decrease or > 500% increase) ")
        else:
            print("    PASS: Percentage changes within expected range")
        
        test_results[season] = season_results
    
    print("\n" + "="*60)
    print("DATA QUALITY TEST SUMMARY")
    print("="*60)
    
    overall_issues = 0
    for season, results in test_results.items():
        issues = []
        if any(results['all_nan_test'].values()):
            issues.append("all-NaN datasets")
        if any(results['negative_values'].values()):
            issues.append("negative values")
        if not all(results['coordinates'].values()):
            issues.append("coordinate inconsistency")
        if results['mwu_test']['coverage_percent'] < 50:
            issues.append("low statistical coverage")
        
        if issues:
            print(f"{season.upper()}: ISSUES FOUND - {', '.join(issues)}")
            overall_issues += len(issues)
        else:
            print(f"{season.upper()}: ALL TESTS PASSED")
    
    if overall_issues == 0:
        print("\nOVERALL: ALL DATA QUALITY TESTS PASSED!")
    else:
        print(f"\nOVERALL: {overall_issues} ISSUES FOUND - Review warnings above")
    
    print("="*60)
    return test_results
print("Sanity check stats function defined.")

Sanity check stats function defined.


In [8]:
# Plotting utility functions
print("\nDefining plotting utility functions...")

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    """Create a subset of an input colormap"""
    new_cmap = colors.LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap

def discrete_cmap(N, base_cmap=None):
    """Create an N-bin discrete colormap from the specified input map"""
    base = plt.cm.get_cmap(base_cmap)
    color_list = base(np.linspace(0, 1, N))
    cmap_name = base.name + str(N)
    return base.from_list(cmap_name, color_list, N)

def create_discretized_colormap_with_arrows_and_ticks(cmap_name, num_bins, vmin, vmax):
    """Create a discretized colormap with over and under arrows"""
    cmap = plt.get_cmap(cmap_name)
    colors_list = cmap(np.linspace(0, 1, num_bins))
    
    colormap_with_arrows = ListedColormap(colors_list[1:-1])
    colormap_with_arrows.set_over(colors_list[-1])
    colormap_with_arrows.set_under(colors_list[0])
    
    tick_positions = np.linspace(vmin, vmax, num_bins - 1)
    return colormap_with_arrows, tick_positions

print("Plotting utility functions defined.")


Defining plotting utility functions...
Plotting utility functions defined.


In [9]:
# Main plotting function
def create_map(ax, cmap, vmin, vmax, data=[], ax_loc_index=0, run_season=[1,2,3,4,5,6,7,8,9,10,11,12], annotations="title only"):
    """Create individual map subplot with detailed progress tracking"""
    print(f"      Creating map for subplot {ax_loc_index}...")
    
    months_dict = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun", 
                   7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"}
    subplot_ltr_dict = {i: chr(ord('a') + i) for i in range(26)}
    
    # Set map extent
    extent = [-125, -73, 24, 50]
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    print(f"        Adding map features...")
    # Add map features
    countries_shp = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_0_countries')
    
    for country, info in zip(shpreader.Reader(countries_shp).geometries(), 
                            shpreader.Reader(countries_shp).records()):
        if info.attributes['NAME_LONG'] != 'United States':
            ax.add_geometries([country], ccrs.PlateCarree(),
                             facecolor='Grey', edgecolor='k', zorder=3)
    
    # Add geographic features
    states_provinces = cfeature.NaturalEarthFeature(
        category='cultural', name='admin_1_states_provinces_lines', scale='50m', facecolor='none')
    ax.add_feature(states_provinces, edgecolor='gray', lw=1., zorder=2)
    ax.add_feature(cfeature.LAND, facecolor=cfeature.COLORS['land_alt1'], zorder=1)
    ax.add_feature(cfeature.OCEAN, zorder=3)
    ax.add_feature(cfeature.COASTLINE, edgecolor='k', lw=.9, alpha=0.8, zorder=2)
    ax.add_feature(cfeature.BORDERS, edgecolor='dimgray', lw=1.2, zorder=2)
    ax.add_feature(cfeature.LAKES, edgecolor='k', lw=.5, zorder=3)
    
    # Plot data
    print(f"        Processing and plotting data...")
    ds_ratio = (21, 21)
    dscrds = downsample_ds(hist_ds['annual'], ds_ratio, "mean")  # Use for coordinates
    
    # uncomment and fix mesh for masked data, see stats section lower in cell
    #data_masked = data[0].where(mask["__xarray_dataarray_variable__"])
    #data_masked_red = downsample_ds(data_masked, ds_ratio, "mean")
    
    #this line jsut to downsample and remove the sim of size 1, (which is how we should be masking)
    data_downsampled = downsample_ds(data[0], ds_ratio, "mean")
    plot_data = data_downsampled['PRECIP_PERIOD'].squeeze()
    
    
    mesh = ax.pcolormesh(dscrds.lon, dscrds.lat, plot_data,
                        cmap=cmap, transform=ccrs.PlateCarree(),
                        shading='auto', vmin=vmin, vmax=vmax, zorder=1)
    

    # Add statistical significance markers if available
    if len(data) > 2:
        print(f"        Adding statistical significance markers...")
        mwu_result_fdr_corrected = fdr_adjust(data[2])

        # Create binary significance mask (threshold defined at top of this file)
        sig_mask_binary = (mwu_result_fdr_corrected.p_value_fdr_corrected < alpha)

        # Wrap in Dataset to preserve structure through downsampling
        sig_mask_ds = xr.Dataset({'p_value_fdr_corrected': sig_mask_binary})

        # Downsample the binary mask - mean gives fraction of significant cells
        stat_sig_fraction = downsample_ds(sig_mask_ds, ds_ratio, "mean")

        # Apply threshold: stipple if ≥X% of constituent cells are significant (threshold set at top of this file)
        stipple_mask = (stat_sig_fraction.p_value_fdr_corrected >= sig_display_threshold)

        # Count significant points
        sig_points = (stipple_mask == True).sum().compute()
        total_points = stipple_mask.size
        print(f"          Adding {sig_points} significant points out of {total_points} total")

        ax.scatter(x=dscrds.lon.where(stipple_mask == True, np.nan),
                   y=dscrds.lat.where(stipple_mask == True, np.nan),
                   color="k", s=.15, transform=ccrs.PlateCarree(), zorder=2)
    
    # Add formatting
    ax.xaxis.set_major_formatter(LONGITUDE_FORMATTER) 
    ax.yaxis.set_major_formatter(LATITUDE_FORMATTER)
    
    # Add subplot label
    anchored_text = AnchoredText(str(subplot_ltr_dict[ax_loc_index]), 
                                loc='upper left', pad=0.3, borderpad=0.2)
    ax.add_artist(anchored_text)
    
    print(f"      Subplot {ax_loc_index} complete.")
    return mesh


In [10]:
# Create the final multi-panel figure
def create_final_figure(departures, percentages, mwu_tests, fdr_tests):
    """Create the final multi-panel precipitation figure with detailed progress tracking"""
    print("\n" + "="*50)
    print("CREATING FINAL MULTI-PANEL FIGURE")
    print("="*50)
    
    print("Setting up colormaps and parameters...")
    # Set up colormap parameters for different plot types
    
    # Historical data (blues/greens)
    print("  Configuring historical data colormap...")
    vmax0, vmin0, c_bins0 = 24.0, 0, 10
    cmap0 = discrete_cmap(c_bins0, truncate_colormap(cmap=colormaps['PuBuGn'], minval=.2, maxval=1, n=100))
    new_cmap0, tick_positions0 = create_discretized_colormap_with_arrows_and_ticks(cmap0, c_bins0, vmin0, vmax0)
    
    # Seasonal historical data
    print("  Configuring seasonal historical data colormap...")
    vmax1, vmin1, c_bins1 = 8.0, 0, 10
    cmap1 = discrete_cmap(c_bins1, truncate_colormap(cmap=colormaps['PuBuGn'], minval=.2, maxval=1, n=100))
    new_cmap1, tick_positions1 = create_discretized_colormap_with_arrows_and_ticks(cmap1, c_bins1, vmin1, vmax1)
    
    # Departure data (diverging)
    print("  Configuring departure data colormap...")
    vmax2, vmin2, c_bins2 = 6, -2, 6
    cmap2 = discrete_cmap(c_bins2, truncate_colormap(cmap=colormaps['BrBG'], minval=.4, maxval=.9, n=100))
    new_cmap2, tick_positions2 = create_discretized_colormap_with_arrows_and_ticks(cmap2, c_bins2, vmin2, vmax2)
    
    # Seasonal departure data
    print("  Configuring seasonal departure data colormap...")
    vmax3, vmin3, c_bins3 = 6.0, -6.0, 10
    cmap3 = discrete_cmap(c_bins3, truncate_colormap(cmap=colormaps['BrBG'], minval=.1, maxval=.9, n=100))
    new_cmap3, tick_positions3 = create_discretized_colormap_with_arrows_and_ticks(cmap3, c_bins3, vmin3, vmax3)
    
    # Percentage data
    print("  Configuring percentage data colormap...")
    vmax4, vmin4, c_bins4 = 200, -100, 8
    cmap4 = discrete_cmap(c_bins4, truncate_colormap(cmap=colormaps['PiYG'], minval=.25, maxval=1, n=100))
    new_cmap4, tick_positions4 = create_discretized_colormap_with_arrows_and_ticks(cmap4, c_bins4, vmin4, vmax4)
    
    # Seasonal percentage data
    print("  Configuring seasonal percentage data colormap...")
    vmax5, vmin5, c_bins5 = 200, -100, 8 
    cmap5 = discrete_cmap(c_bins5, truncate_colormap(cmap=colormaps['PiYG'], minval=.25, maxval=1, n=100))
    new_cmap5, tick_positions5 = create_discretized_colormap_with_arrows_and_ticks(cmap5, c_bins5, vmin5, vmax5)
    
    print("Organizing data for plotting...")
    # Organize data for plotting
    datasets = [
        [hist_ds['annual'].mean(dim="Time")],
        [departures['annual'], [], mwu_tests['annual']],
        [percentages['annual']],
        [hist_ds['mam'].mean(dim="Time")],
        [departures['mam'], [], mwu_tests['mam']],
        [percentages['mam']],
        [hist_ds['jja'].mean(dim="Time")],
        [departures['jja'], [], mwu_tests['jja']],
        [percentages['jja']],
        [hist_ds['son'].mean(dim="Time")],
        [departures['son'], [], mwu_tests['son']],
        [percentages['son']],
        [hist_ds['djf'].mean(dim="Time")],
        [departures['djf'], [], mwu_tests['djf']],
        [percentages['djf']]
    ]
    
    cmaps = [new_cmap0, new_cmap2, new_cmap4,
             new_cmap1, new_cmap3, new_cmap5,
             new_cmap1, new_cmap3, new_cmap5,
             new_cmap1, new_cmap3, new_cmap5,
             new_cmap1, new_cmap3, new_cmap5]
    
    vmaxs = [vmax0, vmax2, vmax4, vmax1, vmax3, vmax5,
             vmax1, vmax3, vmax5, vmax1, vmax3, vmax5, vmax1, vmax3, vmax5]
    
    vmins = [vmin0, vmin2, vmin4, vmin1, vmin3, vmin5,
             vmin1, vmin3, vmin5, vmin1, vmin3, vmin5, vmin1, vmin3, vmin5]
    
    print("Setting up figure layout...")
    # Set up figure and gridspec
    gs = gridspec.GridSpec(8, 3, width_ratios=[4, 4, 4], 
                          height_ratios=[4, .4, 1, 4, 4, 4, 4, .4],
                          top=0.75, bottom=0.1, wspace=0.15, hspace=0.15)
    
    fig = plt.figure(figsize=(10, 14), frameon=True, dpi=600.0)
    print(f"Created figure with size: {fig.get_size_inches()}")
    
    # Create projection
    p = ccrs.LambertConformal(central_longitude=-97.5, central_latitude=38.5, standard_parallels=(38.5, 38.5))
    print("Using Lambert Conformal projection")
    
    print("Creating subplot axes...")
    # Create map axes
    map_axes = []
    colorbar_axes = []
    
    # Row 0 - Annual
    print("  Creating annual row axes...")
    map_axes.extend([fig.add_subplot(gs[0,0], projection=p),
                     fig.add_subplot(gs[0,1], projection=p),
                     fig.add_subplot(gs[0,2], projection=p)])
    
    colorbar_axes.extend([fig.add_subplot(gs[1,0]),
                          fig.add_subplot(gs[1,1]),
                          fig.add_subplot(gs[1,2])])
    
    # Rows 3-6 - Seasonal
    print("  Creating seasonal row axes...")
    for row in [3, 4, 5, 6]:
        map_axes.extend([fig.add_subplot(gs[row,0], projection=p),
                         fig.add_subplot(gs[row,1], projection=p),
                         fig.add_subplot(gs[row,2], projection=p)])
    
    # Last row colorbars
    print("  Creating colorbar axes...")
    colorbar_axes.extend([fig.add_subplot(gs[7,0]),
                          fig.add_subplot(gs[7,1]),
                          fig.add_subplot(gs[7,2])])
    
    # Set extent for all map axes
    extent = [-125, -73, 24, 50]
    print(f"Setting map extent: {extent}")
    for i, ax in enumerate(map_axes):
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    print(f"Created {len(map_axes)} map axes and {len(colorbar_axes)} colorbar axes")
    
    # Create maps
    print("\nCreating individual maps...")
    plot_maps = []
    
    season_names = ['Annual', 'Annual', 'Annual', 'MAM', 'MAM', 'MAM', 
                   'JJA', 'JJA', 'JJA', 'SON', 'SON', 'SON', 'DJF', 'DJF', 'DJF']
    plot_types = ['Historical', 'Departure', 'Percentage'] * 5
    
    for i, (ax, dss, cmap, vn, vx, season, plot_type) in enumerate(
        zip(map_axes, datasets, cmaps, vmins, vmaxs, season_names, plot_types)):
        
        print(f"    Creating map {i+1}/{len(map_axes)}: {season} - {plot_type}")
        plot_map = create_map(ax=ax, cmap=cmap, vmin=vn, vmax=vx, 
                             data=dss, ax_loc_index=i, annotations="no title")
        plot_maps.append(plot_map)
    
    print("\nAdding colorbars...")
    # Add colorbars
    print("  Adding annual colorbars...")
    color_bar10 = plt.colorbar(plot_maps[0], orientation='horizontal', cax=colorbar_axes[0], 
                              extend='max', ticks=tick_positions0[::2], fraction=.8, shrink=.1)
    color_bar10.set_label(r'count', labelpad=3, fontsize=8)
    color_bar10.ax.set_position(color_bar10.ax.get_position().translated(0, 0.01)) # Move up by 1% of the overall figure height (not the subplot height), so it isn't centered (more whitespace underneath)
    
    color_bar11 = plt.colorbar(plot_maps[1], orientation='horizontal', cax=colorbar_axes[1], 
                              extend='both', ticks=tick_positions2[::2], fraction=.8, shrink=.1)
    color_bar11.set_label(r'count Δ', labelpad=3, fontsize=8)
    color_bar11.ax.set_position(color_bar11.ax.get_position().translated(0, 0.01)) 
    
    color_bar12 = plt.colorbar(plot_maps[2], orientation='horizontal', cax=colorbar_axes[2], 
                              extend='max', ticks=tick_positions4[::2], fraction=.8, shrink=.1)
    color_bar12.set_label(r'count Δ (%)', labelpad=3, fontsize=8)
    color_bar12.ax.set_position(color_bar12.ax.get_position().translated(0, 0.01)) 
    
    # Seasonal colorbars
    print("  Adding seasonal colorbars...")
    color_bar70 = plt.colorbar(plot_maps[3], orientation='horizontal', cax=colorbar_axes[3], 
                              extend='max', ticks=tick_positions1[::2], fraction=.8, shrink=.1)
    color_bar70.set_label(r'count', labelpad=3, fontsize=8)
    color_bar70.ax.set_position(color_bar70.ax.get_position().translated(0, 0.01)) 
    color_bar71 = plt.colorbar(plot_maps[4], orientation='horizontal', cax=colorbar_axes[4], 
                              extend='both', ticks=tick_positions3[::2], fraction=.8, shrink=.1)
    color_bar71.set_label(r'count Δ', labelpad=3, fontsize=8)
    color_bar71.ax.set_position(color_bar71.ax.get_position().translated(0, 0.01)) 
    
    color_bar72 = plt.colorbar(plot_maps[5], orientation='horizontal', cax=colorbar_axes[5], 
                              extend='max', ticks=tick_positions5[::2], fraction=.8, shrink=.1)
    color_bar72.set_label(r'count Δ (%)', labelpad=3, fontsize=8)
    color_bar72.ax.set_position(color_bar72.ax.get_position().translated(0, 0.01))
    
    print("Adding titles and labels...")
    RCP_label = RCP_scenario.replace("p",".")
    # Add titles and labels
    map_axes[0].title.set_text(f'HIST Annual Count\n99th percentile occurences')
    map_axes[1].title.set_text(f'EOC{RCP_label} - HIST Annual Count\n99th percentile occurences')
    map_axes[2].title.set_text(f'EOC{RCP_label} - HIST Percentage\n99th percentile occurences')
    
    map_axes[3].title.set_text(f'HIST Seasonal Count\n99th percentile occurences')
    map_axes[4].title.set_text(f'EOC{RCP_label} - HIST Seasonal Count\n99th percentile occurences')
    map_axes[5].title.set_text(f'EOC{RCP_label} - HIST Percentage\n99th percentile occurences')
    
    # Add season labels
    seasons = ['Annual', 'MAM', 'JJA', 'SON', 'DJF']
    season_axes = [map_axes[0], map_axes[3], map_axes[6], map_axes[9], map_axes[12]]
    
    print("  Adding season labels...")
    for ax, season in zip(season_axes, seasons):
        ax.text(-0.07, 0.55, season, va='bottom', ha='center',
                rotation='vertical', rotation_mode='anchor', transform=ax.transAxes)
        print(f"    Added label: {season}")
    
    print("Finalizing layout...")
    plt.tight_layout()
    
    print("="*50)
    print("FIGURE CREATION COMPLETE")
    print("="*50)
    
    return fig

print("multi-panel plot function defined.")

multi-panel plot function defined.


In [ ]:
# Execute the analysis and create figure
def main():
    """Main execution function with comprehensive progress tracking"""
    print("\n" + "="*70)
    print("HYDRO YEAR PRECIPITATION ANALYSIS - MAIN EXECUTION")
    print("="*70)
    
    print("Step 1: Calculating departures and statistical tests...")
    departures, percentages, mwu_tests, fdr_tests = calculate_departures_and_stats()
    
    print("\nStep 2: Running data quality tests...")
    test_results = run_data_quality_tests(hist_ds, futr_ds, departures, percentages, mwu_tests, fdr_tests)
    
    print("\nStep 3: Creating final figure...")
    figure = create_final_figure(departures, percentages, mwu_tests, fdr_tests)
    
    print("\n" + "="*40)
    print("DISPLAYING FIGURE")
    print("="*40)
    
    plt.show()
    
    return figure, test_results, departures, percentages, mwu_tests, fdr_tests

# Execute the main analysis
if __name__ == "__main__":
    results = main()
    figure, test_results, departures, percentages, mwu_tests, fdr_tests = results
    
    #Optional: Uncomment the line below to save the figure
    #figure.savefig('hydro_year_precipitation_analysis.png', dpi=600, bbox_inches='tight')
    
   
    # Print summary of cached statistical files
    stats_files = get_stats_file_paths()
    print("\nSaved statistical files:")
    for season, files in stats_files.items():
        for test_type, filepath in files.items():
            if os.path.exists(filepath):
                file_size = os.path.getsize(filepath) / (1024*1024)  # MB
                print(f"  {season.upper()} {test_type.upper()}: {os.path.basename(filepath)} ({file_size:.1f} MB)")
            else:
                print(f"  {season.upper()} {test_type.upper()}: Not saved")


HYDRO YEAR PRECIPITATION ANALYSIS - MAIN EXECUTION
Step 1: Calculating departures and statistical tests...

STATISTICAL ANALYSIS WITH CACHING
Statistical analysis directory exists: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC8p5/epoch_aggregated/statistical_analysis

Checking for existing statistical files...
  Found existing MWU file for ANNUAL: MWU_HIST-EOC8p5_99th-100th_WetPeriod_0p254base_annual.nc
  Found existing FDR file for ANNUAL: MWU_FDR_HIST-EOC8p5_99th-100th_WetPeriod_0p254base_annual.nc
  Found existing MWU file for MAM: MWU_HIST-EOC8p5_99th-100th_WetPeriod_0p254base_mam.nc
  Found existing FDR file for MAM: MWU_FDR_HIST-EOC8p5_99th-100th_WetPeriod_0p254base_mam.nc
  Found existing MWU file for JJA: MWU_HIST-EOC8p5_99th-100th_WetPeriod_0p254base_jja.nc
  Found existing FDR file for JJA: MWU_FDR_HIST-EOC8p5_99th-100th_WetPeriod_0p254base_jja.nc
  Found existing MWU file for SON: MWU_HIST-EOC8p5_99th-100th_WetPeriod_0p254base_son.nc
  Found existing FDR file

Processing departures/percentages:   0%|          | 0/5 [00:00<?, ?it/s]


  Processing ANNUAL data:
    Calculating temporal means...
    Computing departures (future - historical)...
    Computing percentage changes...
    ANNUAL departures/percentages complete.

  Processing MAM data:
    Calculating temporal means...
    Computing departures (future - historical)...
    Computing percentage changes...
    MAM departures/percentages complete.

  Processing JJA data:
    Calculating temporal means...
    Computing departures (future - historical)...
    Computing percentage changes...
    JJA departures/percentages complete.

  Processing SON data:
    Calculating temporal means...
    Computing departures (future - historical)...
    Computing percentage changes...
    SON departures/percentages complete.

  Processing DJF data:
    Calculating temporal means...
    Computing departures (future - historical)...
    Computing percentage changes...
    DJF departures/percentages complete.

STATISTICAL ANALYSIS COMPLETE

Step 2: Running data quality tests...

/tmp/ipykernel_19009/4147777449.py:13: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  base = plt.cm.get_cmap(base_cmap)


  Creating seasonal row axes...
